# Frozen release: evaluation on future weeks

**Result:** official stability **0.729674**, ROC AUC **0.875759**, Brier **0.019282**
on **203,345 applications from weeks 73-91**.

Before reading these labels, the release froze the 700-feature plan, 90% tuned /
10% original LightGBM weights, no calibration, and development-derived training
budgets (1,852 and 1,355 rounds). The evaluated models were trained on weeks 0-72.
The separate all-label refit is an inference artifact; this score is never attributed
to a model trained on the evaluation labels.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from home_credit.modeling.portfolio import load_portfolio

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
evidence = load_portfolio(root)
print("Verified immutable experiment evidence; no model fitting.")

evaluation = evidence["evaluation"]
display(pd.Series(evaluation["metrics"], name="Reserved-period result").to_frame().round(6))
print("Evaluation cases:", f"{evaluation['rows']:,}")
print("Observed default rate:", f"{evaluation['positive_rate']:.3%}")
print("Frozen evaluation completed (UTC):", evaluation["evaluated_utc"])

## Stability and probability quality

The official formula is mean weekly Gini + 88 x min(weekly slope, 0) - 0.5 x
residual standard deviation. Here the slope is positive, so its penalty is zero.
The weekly plot shows the variation that the overall AUC alone would hide.

Raw probability calibration remains imperfect. The reliability plot is descriptive;
no calibrator is applied to this frozen model. The later development-only study in
notebook 12 neither accesses this holdout nor changes the released bundle. The
holdout base rate differs from development periods, so the lower Brier score alone
is not a model-improvement claim.
A Kaggle leaderboard result has not been obtained.

In [ ]:
from home_credit.modeling.portfolio_report import display_charts

display_charts(evidence, "release")
display(pd.DataFrame(evaluation["weekly"]).round(6))

## Distinct evaluation and inference models

Four native fits completed: two development-only models for the one frozen evaluation,
then two all-label models for inference. Each fit has a verified encoder, exact feature
order, fixed iterations, a reload parity check and a content-addressed S3 checkpoint.
The release was rerun successfully with all completed stages reused.

In [ ]:
stages = evidence["state"]["stages"]
records = []
for name, stage in stages.items():
    if "/" not in name:
        continue
    records.append(
        {
            "Artifact": name,
            "Fit rows": stage["fit_rows"],
            "Fit weeks": str(stage["fit_weeks"]),
            "Iterations": stage["actual_rounds"],
            "Reload error": stage["reload_max_absolute_error"],
        }
    )
display(pd.DataFrame(records))
print("Portable bundle files:", len(stages["bundle"]["files"]))
print("Public example parity cases:", stages["raw_parity"]["rows"])

## What is deployable and what has been tested

The portable bundle contains native LightGBM models, frozen frequency maps,
feature specifications, raw schema and matching inference source. It rebuilds
features from the test files supplied to the run, fingerprints inputs, validates
case coverage and reuses verified prediction batches.

Full raw-to-feature parity was checked on the competition's ten public example
cases. That is an integration fixture, not a hidden-test evaluation or proof of
performance at hidden-test scale. Synthetic tests cover additional missingness,
shard and corruption cases. Notebook 10 gives the owner explicit controls to
generate, validate, save and download a submission; no automatic upload occurs.

This is an evaluated research portfolio release. Production lending use, calibrated
deployment probabilities, fairness assessment and operational monitoring are outside
the validated scope. Completed development feature research and its limits are
documented in [notebook 11](11_feature_research.ipynb).